# Match Backblaze Sampled Rows to Raw Numerical Rows

This notebook matches the sampled Backblaze rows produced by the D2T notebook back to the original daily Backblaze numerical CSV files.

Expected repository layout:

```text
input/
  backblaze/
    original_csv/
      YYYY-MM-DD.csv

output/
  data/
    {disk_model_id}/
      monthly_textual/
        *.csv
      monthly_numerical/
        matched_raw_YYYY-MM.csv
      train.csv
      val.csv
      test.csv
```


In [ ]:
import logging
import re
from datetime import datetime
from multiprocessing import Pool, cpu_count
from pathlib import Path
from typing import Dict, List

import pandas as pd
from tqdm import tqdm


In [ ]:
# =========================
# Path configuration
# =========================
PROJECT_ROOT = Path.cwd()

INPUT_ROOT = PROJECT_ROOT / "input"
OUTPUT_ROOT = PROJECT_ROOT / "output"

DISK_MODEL_ID = "ST14000NM001G"

BACKBLAZE_RAW_DIR = INPUT_ROOT / "backblaze" / "original_csv"

OUTPUT_DATA_DIR = OUTPUT_ROOT / "data" / DISK_MODEL_ID
SAMPLED_TEXTUAL_DIR = OUTPUT_DATA_DIR / "monthly_textual"
MATCHED_MONTHLY_DIR = OUTPUT_DATA_DIR / "monthly_numerical"

OUTPUT_DATA_DIR.mkdir(parents=True, exist_ok=True)
MATCHED_MONTHLY_DIR.mkdir(parents=True, exist_ok=True)

NUM_PROCESSES = max(1, cpu_count() - 1)

print(f"Raw Backblaze CSV dir : {BACKBLAZE_RAW_DIR}")
print(f"Sampled textual dir   : {SAMPLED_TEXTUAL_DIR}")
print(f"Matched monthly dir   : {MATCHED_MONTHLY_DIR}")
print(f"Processes             : {NUM_PROCESSES}")

In [ ]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-7s | %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)

In [ ]:
def extract_month_key(filename: str) -> str:
    """
    Extract YYYY-MM from a sampled textual filename.

    Supported examples:
    - backblaze_textual_2024-01-01_to_2024-01-31.csv
    - RLFH-2024-01.csv
    - any filename containing YYYY-MM
    """
    match = re.search(r"(20\d{2})-(\d{2})", filename)
    if match is None:
        return "unknown"
    return f"{match.group(1)}-{match.group(2)}"


def load_label_keys(label_path: Path) -> pd.DataFrame:
    """
    Load the sampled rows that should be matched back to raw numerical rows.

    Required columns:
    - date
    - serial_number

    Optional label columns:
    - failure_7d
    - upcoming_failure

    The returned dataframe is one row per sampled (date, serial_number).
    """
    header = pd.read_csv(label_path, nrows=0)
    available_cols = set(header.columns)

    required_cols = ["date", "serial_number"]
    missing = [col for col in required_cols if col not in available_cols]
    if missing:
        raise ValueError(f"{label_path.name} is missing required columns: {missing}")

    optional_cols = [col for col in ["failure_7d", "upcoming_failure"] if col in available_cols]
    usecols = required_cols + optional_cols

    labels = pd.read_csv(
        label_path,
        usecols=usecols,
        dtype={"date": "string", "serial_number": "string"},
        low_memory=False,
    )

    labels = labels.dropna(subset=["date", "serial_number"]).copy()
    labels["date"] = pd.to_datetime(labels["date"], errors="coerce").dt.strftime("%Y-%m-%d")
    labels["serial_number"] = labels["serial_number"].astype(str).str.strip()
    labels = labels.dropna(subset=["date", "serial_number"])
    labels = labels.drop_duplicates(subset=["date", "serial_number"])

    return labels


def get_needed_serials_per_day(labels: pd.DataFrame) -> Dict[str, set]:
    """
    Return a dictionary:
    {
      "2024-05-17": {"ABC123", "DEF456", ...},
      ...
    }
    """
    return labels.groupby("date")["serial_number"].apply(set).to_dict()


In [ ]:
def process_one_month(label_path: Path) -> int:
    """
    Match one sampled textual monthly CSV back to original daily Backblaze rows.
    The output is saved as:
    output/data/{disk_model_id}/monthly_numerical/matched_raw_YYYY-MM.csv
    """
    month_key = extract_month_key(label_path.name)
    if month_key == "unknown":
        logging.warning(f"Cannot parse month from {label_path.name}; skipping.")
        return 0

    output_path = MATCHED_MONTHLY_DIR / f"matched_raw_{month_key}.csv"

    labels = load_label_keys(label_path)
    required = get_needed_serials_per_day(labels)

    if not required:
        logging.info(f"{label_path.name} -> no sampled rows.")
        return 0

    matched_chunks = []

    for date_str, serial_set in required.items():
        raw_path = BACKBLAZE_RAW_DIR / f"{date_str}.csv"
        if not raw_path.is_file():
            logging.warning(f"Raw daily file not found: {raw_path}")
            continue

        try:
            df_day = pd.read_csv(raw_path, low_memory=False)
            if "date" not in df_day.columns or "serial_number" not in df_day.columns:
                raise ValueError("Raw file must contain date and serial_number columns.")

            df_day["date"] = pd.to_datetime(df_day["date"], errors="coerce").dt.strftime("%Y-%m-%d")
            df_day["serial_number"] = df_day["serial_number"].astype(str).str.strip()

            day_rows = df_day[df_day["serial_number"].isin(serial_set)].copy()
            if day_rows.empty:
                continue

            day_labels = labels[labels["date"] == date_str].copy()
            day_rows = day_rows.merge(
                day_labels,
                on=["date", "serial_number"],
                how="inner",
                suffixes=("", "_label"),
            )

            if not day_rows.empty:
                matched_chunks.append(day_rows)

        except Exception as e:
            logging.error(f"Error reading {raw_path.name}: {e}")

    if not matched_chunks:
        logging.info(f"{label_path.name} -> no matched rows.")
        return 0

    df_final = pd.concat(matched_chunks, ignore_index=True)

    # Ensure failure_7d exists. Positive samples include both:
    # - rows whose sampled-text label says upcoming failure is Yes
    # - rows whose raw current-date failure column is already positive
    if "failure_7d" not in df_final.columns:
        upcoming_positive = (
            df_final["upcoming_failure"].astype(str).str.startswith("Yes")
            if "upcoming_failure" in df_final.columns
            else False
        )
        current_failure = (
            pd.to_numeric(df_final["failure"], errors="coerce").fillna(0).astype(int) == 1
            if "failure" in df_final.columns
            else False
        )
        df_final["failure_7d"] = (upcoming_positive | current_failure).astype(int)
    else:
        current_failure = (
            pd.to_numeric(df_final["failure"], errors="coerce").fillna(0).astype(int) == 1
            if "failure" in df_final.columns
            else False
        )
        df_final["failure_7d"] = (
            pd.to_numeric(df_final["failure_7d"], errors="coerce").fillna(0).astype(int).eq(1)
            | current_failure
        ).astype(int)

    df_final.to_csv(output_path, index=False)

    matched_count = len(df_final)
    logging.info(f"{label_path.name} -> {matched_count:,} rows -> {output_path.name}")
    return matched_count


In [ ]:
def match_all_months() -> None:
    start_time = datetime.now()

    label_files = sorted(SAMPLED_TEXTUAL_DIR.glob("*.csv"))
    if not label_files:
        raise FileNotFoundError(f"No sampled textual CSV files found under: {SAMPLED_TEXTUAL_DIR}")

    print(f"Start          : {start_time.strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"Disk model     : {DISK_MODEL_ID}")
    print(f"Sampled files  : {len(label_files)}")
    print(f"Processes      : {NUM_PROCESSES}")
    print("-" * 70)

    with Pool(processes=NUM_PROCESSES) as pool:
        results = list(tqdm(
            pool.imap(process_one_month, label_files),
            total=len(label_files),
            desc=f"{DISK_MODEL_ID} monthly matching",
        ))

    total_rows = sum(results)
    duration = datetime.now() - start_time

    print("\n" + "=" * 70)
    print("SUMMARY")
    print("-" * 70)
    print(f"Processed files: {len(label_files)}")
    print(f"Matched rows   : {total_rows:,}")
    print(f"Duration       : {duration}")
    print("=" * 70)


match_all_months()


## Save train / validation / test numerical files

The matched monthly numerical rows are combined and saved under:

```text
output/data/{disk_model_id}/train.csv
output/data/{disk_model_id}/val.csv
output/data/{disk_model_id}/test.csv
```

For Backblaze, the split follows the experiment protocol:

```text
train = 2024-01-01 to 2025-03-31
val   = 2025-01-01 to 2025-03-31
test  = 2025-04-01 to 2025-09-30
```

Here `val` is a threshold-calibration subset inside the training/development period.


In [ ]:
def save_train_val_test_from_matched_monthly() -> None:
    matched_files = sorted(MATCHED_MONTHLY_DIR.glob("matched_raw_*.csv"))
    if not matched_files:
        raise FileNotFoundError(f"No matched monthly files found under: {MATCHED_MONTHLY_DIR}")

    df = pd.concat(
        [pd.read_csv(path, low_memory=False) for path in matched_files],
        ignore_index=True,
    )

    if "date" not in df.columns:
        raise ValueError("Matched dataframe must contain a date column.")

    if "serial_number" not in df.columns:
        raise ValueError("Matched dataframe must contain a serial_number column.")

    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df = df.dropna(subset=["date", "serial_number"]).copy()
    df = df.sort_values(["date", "serial_number"]).reset_index(drop=True)

    if "failure_7d" not in df.columns:
        raise ValueError("Matched dataframe must contain failure_7d. Check the monthly matching step.")

    TRAIN_START = pd.Timestamp("2024-01-01")
    TRAIN_END = pd.Timestamp("2025-03-31")

    VAL_START = pd.Timestamp("2025-01-01")
    VAL_END = pd.Timestamp("2025-03-31")

    TEST_START = pd.Timestamp("2025-04-01")
    TEST_END = pd.Timestamp("2025-09-30")

    train_df = df[
        (df["date"] >= TRAIN_START)
        & (df["date"] <= TRAIN_END)
    ].copy()

    val_df = df[
        (df["date"] >= VAL_START)
        & (df["date"] <= VAL_END)
    ].copy()

    test_df = df[
        (df["date"] >= TEST_START)
        & (df["date"] <= TEST_END)
    ].copy()

    if train_df.empty:
        raise ValueError(f"Empty train split: {TRAIN_START.date()} to {TRAIN_END.date()}")

    if val_df.empty:
        raise ValueError(f"Empty validation split: {VAL_START.date()} to {VAL_END.date()}")

    if test_df.empty:
        raise ValueError(f"Empty test split: {TEST_START.date()} to {TEST_END.date()}")

    train_df.to_csv(OUTPUT_DATA_DIR / "train.csv", index=False)
    val_df.to_csv(OUTPUT_DATA_DIR / "val.csv", index=False)
    test_df.to_csv(OUTPUT_DATA_DIR / "test.csv", index=False)

    print(f"Saved train.csv: {train_df.shape} [{TRAIN_START.date()} to {TRAIN_END.date()}]")
    print(f"Saved val.csv  : {val_df.shape} [{VAL_START.date()} to {VAL_END.date()}]")
    print(f"Saved test.csv : {test_df.shape} [{TEST_START.date()} to {TEST_END.date()}]")


save_train_val_test_from_matched_monthly()


In [ ]:
# Preview generated outputs.
print("Matched monthly files:")
for path in sorted(MATCHED_MONTHLY_DIR.glob("matched_raw_*.csv"))[:5]:
    print(f"  {path}")

print("\nFinal numerical split files:")
for name in ["train.csv", "val.csv", "test.csv"]:
    path = OUTPUT_DATA_DIR / name
    print(f"  {path} -> exists={path.exists()}")
